In [ ]:
import torch

# データのバッチ処理やシャッフルを管理するユーティリティモジュール
# TensorDataset: 特徴量(X)とラベル(y)などのテンソルを1つのデータセットにまとめるクラス
# DataLoader: データセットから指定したバッチサイズで自動的にデータを取得・シャッフルするクラス
from torch.utils.data import TensorDataset, DataLoader


In [ ]:
# 特徴量データ X（入力）：形状 [8, 1]（8件のデータ、各データの特徴量は1つ）
X = torch.tensor([
    [1.0],
    [2.0],
    [3.0],
    [4.0],
    [5.0],
    [6.0],
    [7.0],
    [8.0]
])

# 目的変数データ y（正解ラベル）：形状 [8, 1]（y = 2x + 1 の関係性）
y = torch.tensor([
    [3.0],
    [5.0],
    [7.0],
    [9.0],
    [11.0],
    [13.0],
    [15.0],
    [17.0]
])

# 各テンソルの形状（次元・サイズ）を確認
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: torch.Size([8, 1])
y shape: torch.Size([8, 1])


In [ ]:
# 特徴量 X と正解ラベル y をペアにして 1 つの Dataset オブジェクトに統合
dataset = TensorDataset(X, y)

# Dataset オブジェクトの参照（クラス名とメモリ領域）を出力
print(dataset)
# Dataset に含まれるデータサンプル数（全件数 = 8）を確認
print("Number of samples:", len(dataset))

Number of samples: 8


In [ ]:
# dataset[0] で 0 番目のデータサンプルを取得（(X[0], y[0]) のタプルとして取り出し）
sample_X, sample_y = dataset[0]

# 取り出した 0 番目の特徴量 X と正解ラベル y を確認（形状は 1 次元テンソル [1] になる）
print("X:", sample_X)
print("y:", sample_y)

X: tensor([1.])
y: tensor([3.])


In [ ]:
# dataset[3] で 3 番目（通算 4 件目）のデータペア (X[3], y[3]) をタプルとして直接取得して表示
print(dataset[3])

(tensor([4.]), tensor([9.]))


In [ ]:
# Dataset からデータをミニバッチ（小分け）単位で取り出す DataLoader を生成
dataloader = DataLoader(
    dataset,　# 対象の Dataset オブジェクト
    batch_size=2,　# 1回のイテレーション（ループ）でまとめて取り出すデータ件数（2件ずつ）
    shuffle=False　# データを取り出す順序をシャッフルせず、インデックス順（元の並び順）のまま取得する設定
)

In [ ]:
# DataLoaderからミニバッチ単位（batch_size=2）でデータを取り出してループ処理
for X_batch, y_batch in dataloader:

    # 取得したバッチデータの特徴量 X_batch（形状: [2, 1]）とラベル y_batch（形状: [2, 1]）を出力
    print("X batch:", X_batch)
    print("y batch:", y_batch)

    # y_batch を再度確認のため別フォーマット（改行あり）で出力
    print("y batch:")
    print(y_batch)

    # 各バッチの処理結果の区切り線を出力
    print("---")

X batch: tensor([[1.],
        [2.]])
y batch: tensor([[3.],
        [5.]])
y batch:
tensor([[3.],
        [5.]])
---
X batch: tensor([[3.],
        [4.]])
y batch: tensor([[7.],
        [9.]])
y batch:
tensor([[7.],
        [9.]])
---
X batch: tensor([[5.],
        [6.]])
y batch: tensor([[11.],
        [13.]])
y batch:
tensor([[11.],
        [13.]])
---
X batch: tensor([[7.],
        [8.]])
y batch: tensor([[15.],
        [17.]])
y batch:
tensor([[15.],
        [17.]])
---


In [ ]:
# バッチサイズを 4 件に変更して DataLoader を再定義（全8件のデータを 4件×2回 で処理）
# shuffle=False のため、インデックス順（1〜4件目、5〜8件目）に取得される
dataloader = DataLoader(dataset, batch_size=4, shuffle=False)

In [ ]:
# バッチサイズ 4 に設定した dataloader からデータを取り出してループ処理
for X_batch, y_batch in dataloader:
    # 取得したミニバッチの特徴量 X_batch（形状: [4, 1]）を表示
    print("X batch:", X_batch)
    # 各バッチの区切り線を出力（全8件のデータが 4件ずつ 2 回に分けて処理される）
    print("---")

X batch: tensor([[1.],
        [2.],
        [3.],
        [4.]])
---
X batch: tensor([[5.],
        [6.],
        [7.],
        [8.]])
---


In [ ]:
# shuffle=True に変更して DataLoader を再定義
# 各エポックの開始時にデータの順番をランダムにシャッフルし、2件ずつ（全4バッチ）取得する
# ※学習時にシャッフルを入れることで、データの並び順への依存（バイアス）や過学習を防ぐ
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

In [ ]:
# シャッフルされた dataloader からミニバッチごとに処理を実行
for X_batch, y_batch in dataloader:
    # X_batch (形状: [2, 1]) の無駄な次元を .squeeze() で削って 1 次元化 ([2]) し、
    # .tolist() で PyTorch テンソルから Python の通常のリスト型（例: [2.0, 7.0]）に変換して出力
    print(X_batch.squeeze().tolist())

[8.0, 7.0]
[5.0, 4.0]
[1.0, 2.0]
[3.0, 6.0]


In [ ]:
import torch.nn as nn　# ニューラルネットワーク構築に必要な基本モジュール群をインポート

# PyTorch のモデルクラスを定義（nn.Module を継承することが必須ルール）
class LinearModel(nn.Module):

    def __init__(self):
        # 親クラス（nn.Module）の初期化処理を呼び出し（モデルのパラメータ管理に必要な設定が行われる）
        super().__init__()

        # 線形結合層（全結合層）を定義（入力特徴量数: 1、出力特徴量数: 1）
        # 内部で重み (weight) とバイアス (bias) がランダムな初期値で生成される
        self.linear = nn.Linear(1, 1)

    # 順伝播（Forward Pass）処理の定義
    # model(x) を実行した際に自動的にこの forward メソッドが呼び出される
    def forward(self, x):
        # 入力 x を線形層に通して計算結果（y = wx + b）を返す
        return self.linear(x)

In [ ]:
# 乱数シードを 42 に固定し、重みの初期化やデータシャッフルの再現性を確保
torch.manual_seed(42)

# 定義した LinearModel クラスのインスタンス（モデル本体）を作成
model = LinearModel()

# 損失関数に平均二乗誤差（MSE: Mean Squared Error）を設定（回帰問題の標準的な評価指標）
loss_fn = nn.MSELoss()

# 最適化アルゴリズムとして確率的勾配降下法（SGD）を設定
# model.parameters(): 最適化対象（更新する重み・バイアス）を渡す
# lr=0.01: パラメータを一度に更新するステップ幅（学習率: Learning Rate）
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# 学習用の DataLoader を設定（2件ずつのミニバッチを取り出し、各エポックでシャッフル）
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# データセット全体を使った学習の繰り返し回数（エポック数）を 100 回に設定
epochs = 100

/opt/anaconda3/lib/python3.12/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(


In [ ]:
# 指定したエポック数（100回）だけデータセット全体の学習を繰り返すメインループ
for epoch in range(epochs):
    # 1エポック内の全バッチの損失値を合計するための変数を初期化
    total_loss = 0.0
    # DataLoader からミニバッチ単位（X_batch: [2, 1], y_batch: [2, 1]）でデータを取り出してループ処理
    for X_batch, y_batch in dataloader:
        # 1. 前回のバッチ計算で溜まった勾配（微分値）を 0 にリセット（勾配の蓄積・加算を防ぐ）
        optimizer.zero_grad()
        # 2. 順伝播（Forward）：ミニバッチデータをモデルに入力し、予測値を取得
        predictions = model(X_batch)
        # 3. 損失計算：モデルの予測値と実際の正解ラベルとの間の誤差（MSE）を算出
        loss = loss_fn(predictions, y_batch)
        # 4. 逆伝播（Backward）：計算グラフを遡り、各パラメータの勾配（微分値）を計算して .grad に格納
        loss.backward()
        # 5. パラメータ更新：算出された勾配と学習率（lr=0.01）に基づき、重みとバイアスを更新（w = w - lr * g）
        optimizer.step()

        # ミニバッチの損失値（PyTorchスカラテンソル）を .item() で Python の float 値に変換して加算
        # （計算グラフから切り離すことで、無駄なメモリの蓄積を防ぐ）
        total_loss += loss.item()

    # 10 エポックごとに、そのエポック内での合計損失（total_loss）を出力して学習経過を確認
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.6f}")

Epoch 0, Loss: 57.874990
Epoch 10, Loss: 0.000268
Epoch 20, Loss: 0.000222
Epoch 30, Loss: 0.000151
Epoch 40, Loss: 0.000102
Epoch 50, Loss: 0.000075
Epoch 60, Loss: 0.000060
Epoch 70, Loss: 0.000041
Epoch 80, Loss: 0.000028
Epoch 90, Loss: 0.000020


In [ ]:
# model.named_parameters() でモデル内のすべての学習対象パラメータの「名前」と「テンソル」を取得
# name: パラメータ名（例: 'linear.weight', 'linear.bias'）
# param: パラメータオブジェクト（torch.nn.Parameter）
for name, param in model.named_parameters():
    # param.data で勾配計算情報（grad_fn）を除いたパラメータの純粋なテンソルデータ（値）を表示
    # 学習が正常に進んでいれば、y = 2x + 1 の関係に従い、
    # linear.weight は [[2.0]] 付近、linear.bias は [1.0] 付近に収束している
    print(name, param.data)

linear.weight tensor([[1.9993]])
linear.bias tensor([1.0040])


In [ ]:
# 推論テスト用の新しい入力データを準備（形状: [1, 1] の 2 次元テンソル）
# ※学習時と同じく (batch_size, in_features) の形状に揃える必要がある
new_x = torch.tensor([[10.0]])

# 自動微分（勾配計算）に必要な記憶領域の確保・計算グラフ構築を一時的にオフにするコンテキストマネージャ
# ※推論（評価）時に使用することで、メモリ消費を抑え、計算スピードを高速化させる
with torch.no_grad():
    # 学習済みモデルに未知の入力データ new_x を渡して順伝播を行い、予測値を取り出す
    prediction = model(new_x)
# 計算結果を表示
# 学習結果のパラメータ y ≈ 2x + 1 に従い、x=10.0 に対する予測値（tensor([[21.0000]]) 付近）が出力される
print(prediction)

tensor([[20.9975]])
